# Модуль 6.4 — LLM API на практике (прямой SDK)

Домашка к [лекции 6.4](https://itrubnikov.github.io/Train_of_Thought/docs/modules/06-4-llm-api/). Вы соберёте три утилиты на реальном API: summarizer (streaming), классификатор писем (structured output) и переводчик-редактор (многоходовый диалог). Нужен API-ключ — см. README.

## 0. Установка и клиент

Запустите ячейку. В Colab ключ берётся из Secrets (значок 🔑 слева, имя `ANTHROPIC_API_KEY`); локально — из `.env` (скопируйте `.env.example`).

In [ ]:
!pip -q install anthropic openai python-dotenv pydantic
import os

# ключ: Colab Secrets -> переменная окружения; иначе .env
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    from dotenv import load_dotenv
    load_dotenv()

if not os.getenv("ANTHROPIC_API_KEY"):
    raise RuntimeError("Нет ANTHROPIC_API_KEY. Впишите его в .env (см. .env.example) или в Colab Secrets.")

from anthropic import Anthropic
client = Anthropic()                 # читает ANTHROPIC_API_KEY из окружения
MODEL = "claude-haiku-4-5"           # дешёвый для учёбы; флагман — "claude-opus-4-8"
print("Готово, клиент создан.")

## 1. summarizer со streaming

Готовая функция `summarize(text)`: зовёт `client.messages.stream(...)`, печатает
токены потоком и возвращает текст. Запусти и смотри, как конспект появляется по
мере генерации (streaming — это про отзывчивость, не про скорость). Покрутить —
в задачах в конце.

In [ ]:
ARTICLE = (
    "Трансформеры вытеснили рекуррентные сети, потому что механизм attention "
    "позволяет обрабатывать всю последовательность параллельно, а не по шагам. "
    "Это дало масштабируемость на GPU и способность ловить дальние связи в тексте. "
    "На этой архитектуре выросли все современные большие языковые модели."
)

def summarize(text: str) -> str:
    with client.messages.stream(
        model=MODEL, max_tokens=1024,
        system="Ты делаешь сжатые конспекты. 3-5 пунктов, по-русски.",
        messages=[{"role": "user", "content": f"Сделай конспект:\n\n{text}"}],
    ) as stream:
        for chunk in stream.text_stream:     # токены идут потоком
            print(chunk, end="", flush=True)
        final = stream.get_final_message()
    print()
    print("usage:", final.usage)
    return final.content[0].text

_ = summarize(ARTICLE)

## 2. Классификатор писем — Build-twice

Сначала наивная версия через `json.loads` (в `try/except` — увидишь, устойчива
ли она), потом надёжная через structured output. Обе готовы — запусти и сравни.
Как только ответ читает программа, а не человек, свободный текст превращается в
источник багов; схема (`output_format`) делает из него контракт.

In [ ]:
import json
from typing import Literal
from pydantic import BaseModel

EMAIL = "Здравствуйте! Не пришёл счёт за март, а оплатить надо сегодня. Помогите срочно."

# --- Заход 1: в лоб. json.loads на свободном тексте хрупкий -> оборачиваем в try ---
def classify_naive(email: str) -> dict:
    resp = client.messages.create(
        model=MODEL, max_tokens=300,
        messages=[{"role": "user", "content":
            f"Верни JSON с полями category, urgency, needs_reply для письма:\n\n{email}"}],
    )
    return json.loads(resp.content[0].text)

try:
    print("наивный результат:", classify_naive(EMAIL))
except Exception as e:
    print("наивный json.loads упал:", type(e).__name__, "-", e)
    print("(вот ради этого и нужен structured output ниже)")

# --- Заход 2: по-инженерному. structured output гарантирует схему ---
class EmailLabel(BaseModel):
    category: Literal["спам", "счёт", "поддержка", "личное", "другое"]
    urgency: Literal["низкая", "средняя", "высокая"]
    needs_reply: bool

def classify(email: str) -> EmailLabel:
    resp = client.messages.parse(
        model=MODEL, max_tokens=512,
        messages=[{"role": "user", "content": f"Классифицируй письмо:\n\n{email}"}],
        output_format=EmailLabel,
    )
    return resp.parsed_output

print("structured:", classify(EMAIL))

## 3. Переводчик-редактор — многоходовый диалог

Готовая функция: переводит фразу, затем правит её по второй реплике. Видно, что
историю диалога ведёшь ты сам (API stateless) — ответ модели мы вручную кладём
обратно в `messages`. Запусти.

In [ ]:
SYSTEM = "Ты переводчик-редактор: переводишь на русский и улучшаешь стиль."

def translate_then_edit(phrase: str, instruction: str) -> str:
    messages = [{"role": "user", "content": f"Translate: {phrase!r}"}]
    r1 = client.messages.create(model=MODEL, max_tokens=512, system=SYSTEM, messages=messages)
    print("перевод:", r1.content[0].text)
    messages.append({"role": "assistant", "content": r1.content})  # <- память диалога ведём мы
    messages.append({"role": "user", "content": instruction})
    r2 = client.messages.create(model=MODEL, max_tokens=512, system=SYSTEM, messages=messages)
    return r2.content[0].text

print("правка:", translate_then_edit("The cat sat on the mat.", "Сделай официальнее."))

## Задачи — доработайте рабочий код

Три утилиты работают. Теперь учимся, меняя готовое:

1. **Свой текст.** Подставь в summarizer свою длинную статью; смени `system` на
   «выжимка в одну фразу» — как изменился конспект?
2. **Сломай наивный парсер.** Подбери письмо, на котором `classify_naive` падает
   или врёт, а `classify` (structured) — нет. Запиши пример и вывод.
3. **Новое поле.** Добавь в `EmailLabel` поле `language` (`Literal["ru","en","other"]`)
   и убедись, что structured output его заполняет без других правок.
4. **Забывчивость.** В переводчике убери строку, кладущую ответ в `messages`, и
   повтори — увидишь, что модель «забыла» перевод (это и есть stateless).
5. **(advanced) OpenAI-двойник.** Перепиши любую из трёх утилит на OpenAI SDK по
   двойникам из лекции (`chat.completions.create` / `chat.completions.parse`).

Каждая задача — правка рабочего кода. По каждой запиши короткий вывод.

## Что сдать

- [ ] Ноутбук прогнан целиком (`Run all`) — три утилиты отработали, вывод виден.
- [ ] Найден и записан пример, где наивный `json.loads` сломался/соврал, а structured output — нет.
- [ ] Сделаны задачи-доработки (мин. 3 из 5) с короткими выводами.
- [ ] Ключ не захардкожен (только `.env` / Secrets).

Вывод одной фразой запиши в ячейку ниже: что удивило больше всего.

_(Ваш вывод одной фразой здесь.)_